In [ ]:
# A virtual Newtonian catalog of wide binaries is generated by replacing the observed Gaia proper motions with Monte Carlo simulated values under Newtonian dynamics. One can choose circular orbits or elliptical orbits. For the latter, eccentricities from Hwang et al. (2022) are used.
# The input file 'gaia_dr3_MSMS_d200pc.csv' was extracted from the El-Badry, Rix, & Heintz (2021) catalog.
# A mass-magnitude relation is needed. Three options are possible.
# Dust extinctions are set to zero for virtual binaries.
# The user needs to choose a value for the higher-order multiplicity fraction f_multi. E.g., f_multi = 0.5 means that 50% of binaries are randomly assigned close hidden companions for either component or for both components.
# For further information see: Chae, K.-H. 2023, to be published in the Astrophysical Journal (arXiv:2305.04613)
# If you use this code, please cite the above reference.
# Questions can be directed to kyuhyunchae@gmail.com or chae@sejong.ac.kr.
# Last modified 2023-06-21.
import numpy as np
from scipy.optimize import minimize
import scipy.integrate as integrate
import multiprocessing as mp
import time

def dsq(p,*args):
    phi = p
    t, eps = args
    f = lambda x: 1/(1+eps*np.cos(x))**2
    val, err = integrate.quad(f, 0, phi)
    res = (t-val)**2
    return res

mod = input('model(c or e) = ')
ML_choice = input('Mag-Mass relation (v (V-band), j (J-band), or f (flame+J-band)) = ')
f_multi = float(input('fraction of high-order multiples = '))
num = input('num (output file number) = ')
if mod == 'c':
    model = 'circular'
elif mod == 'e':
    model = 'eccentric'
else:
    model = 'eccentric'  # default choice
if ML_choice == 'v':
    ML = 'M_MagG_polyfit_V_Ic.dat'
    LM = 'MagG_M_polyfit_V_Ic.dat'
elif ML_choice == 'j':
    ML = 'M_MagG_polyfit_J_Bp_Rp.dat'
    LM = 'MagG_M_polyfit_J_Bp_Rp.dat'
elif ML_choice == 'f':
    ML = 'M_MagG_polyfit_F.dat'
    LM = 'MagG_M_polyfit_F.dat'
else:
    ML = 'M_MagG_polyfit_V_Ic.dat'  # defaul choice
    LM = 'MagG_M_polyfit_V_Ic.dat'

aML = np.loadtxt(ML,skiprows=1,unpack=True,usecols=(1),dtype=float)

inputfile = 'gaia_dr3_MSMS_d200pc.csv'  
outputfile = 'Newton_dr3_MSMS_d200pc_'+num+'.csv'

source_id_A, source_id_B = np.loadtxt(inputfile,skiprows=1,delimiter=',',unpack=True,usecols=(0,1),dtype=str)
xxx_A, xxx_B, R_chance, rp, d_A, d_A_err, d_B, d_B_err, MagG_A, MagG_B, M_A, M_B, mux_A, mux_A_err, muy_A, muy_A_err, mux_B, mux_B_err, muy_B, muy_B_err, RV_A, RV_A_err, RV_B, RV_B_err, gal_b, ruwe_A, ruwe_B, bp_rp_A, bp_rp_B, ra_A, dec_A, ra_B, dec_B, e, e0, e1, A_G_A, A_G_B = np.loadtxt(inputfile,skiprows=1,delimiter=',',unpack=True,dtype=float)
A_G_A, A_G_B = 0*A_G_A, 0*A_G_B

d_M = (d_A/d_A_err**2+d_B/d_B_err**2)/(1/d_A_err**2+1/d_B_err**2)
ruwe_bin = np.maximum(ruwe_A, ruwe_B)
d_max = 200
ruwe_max = np.inf
R_max = np.inf
mask0 = (d_M < d_max) & (R_chance < R_max) & (ruwe_bin < ruwe_max) 
source_id_A, source_id_B, R_chance, rp, d_A, d_A_err, d_B, d_B_err, MagG_A, MagG_B, M_A, M_B, mux_A, mux_A_err, muy_A, muy_A_err, mux_B, mux_B_err, muy_B, muy_B_err, RV_A, RV_A_err, RV_B, RV_B_err, gal_b, ruwe_A, ruwe_B, bp_rp_A, bp_rp_B, ra_A, dec_A, ra_B, dec_B, e, e0, e1, A_G_A, A_G_B = source_id_A[mask0], source_id_B[mask0], R_chance[mask0], rp[mask0], d_A[mask0], d_A_err[mask0], d_B[mask0], d_B_err[mask0], MagG_A[mask0], MagG_B[mask0], M_A[mask0], M_B[mask0], mux_A[mask0], mux_A_err[mask0], muy_A[mask0], muy_A_err[mask0], mux_B[mask0], mux_B_err[mask0], muy_B[mask0], muy_B_err[mask0], RV_A[mask0], RV_A_err[mask0], RV_B[mask0], RV_B_err[mask0], gal_b[mask0], ruwe_A[mask0], ruwe_B[mask0], bp_rp_A[mask0], bp_rp_B[mask0], ra_A[mask0], dec_A[mask0], ra_B[mask0], dec_B[mask0], e[mask0], e0[mask0], e1[mask0], A_G_A[mask0], A_G_B[mask0]
ruwe_bin, d_M = ruwe_bin[mask0], d_M[mask0]
mux_M, muy_M = (mux_A+mux_B)/2, (muy_A+muy_B)/2
M_A_1, M_A_2, M_B_1, M_B_2 = np.zeros(len(M_A)), np.zeros(len(M_A)), np.zeros(len(M_A)), np.zeros(len(M_A))

gam = -0.7
gam += 1
for i in range(len(MagG_A)):
    np.random.seed(int(str(int(time.time()*1.e7))[-7:]))
    MG1 = MagG_A[i]
    MG2 = MagG_B[i]
    mn = np.array([MG1**j for j in range(11)]) 
    M_A[i] = 10**np.sum(aML*mn)
    mn = np.array([MG2**j for j in range(11)]) 
    M_B[i] = 10**np.sum(aML*mn)
    p_multi = np.random.rand()
    if p_multi > f_multi: 
        M_A_1[i] = M_A[i]
        M_B_1[i] = M_B[i]
    else:
        p_multi_option = np.random.rand()
        if p_multi_option > 0.6:  # A has an additional component
            delm = 12*np.random.power(gam,1)[0]
            fL1 = 1/(1+10**(-0.4*delm))
            MG1 = -2.5*np.log10(fL1) + MagG_A[i]
            MG2 = -2.5*np.log10(np.maximum(1-fL1,1.e-20)) + MagG_A[i]
            mn = np.array([MG1**j for j in range(11)]) 
            M_A_1[i] = 10**np.sum(aML*mn)
            mn = np.array([MG2**j for j in range(11)]) 
            M_A_2[i] = 10**np.sum(aML*mn)
            M_A[i] = M_A_1[i]+M_A_2[i]
            M_B_1[i] = M_B[i]
        elif p_multi_option > 0.3:  # B has an additional component
            delm = 12*np.random.power(gam,1)[0]
            fL1 = 1/(1+10**(-0.4*delm))
            MG1 = -2.5*np.log10(fL1) + MagG_B[i]
            MG2 = -2.5*np.log10(np.maximum(1-fL1,1.e-20)) + MagG_B[i]
            mn = np.array([MG1**j for j in range(11)]) 
            M_B_1[i] = 10**np.sum(aML*mn)
            mn = np.array([MG2**j for j in range(11)]) 
            M_B_2[i] = 10**np.sum(aML*mn)
            M_B[i] = M_B_1[i]+M_B_2[i]
            M_A_1[i] = M_A[i]            
        else: # Both A and B have additional components respectively
            delm = 12*np.random.power(gam,1)[0]
            fL1 = 1/(1+10**(-0.4*delm))
            MG1 = -2.5*np.log10(fL1) + MagG_A[i]
            MG2 = -2.5*np.log10(np.maximum(1-fL1,1.e-20)) + MagG_A[i]
            mn = np.array([MG1**j for j in range(11)]) 
            M_A_1[i] = 10**np.sum(aML*mn)
            mn = np.array([MG2**j for j in range(11)]) 
            M_A_2[i] = 10**np.sum(aML*mn)
            M_A[i] = M_A_1[i]+M_A_2[i]
            delm = 12*np.random.power(0.3,1)[0]
            fL1 = 1/(1+10**(-0.4*delm))
            MG1 = -2.5*np.log10(fL1) + MagG_B[i]
            MG2 = -2.5*np.log10(np.maximum(1-fL1,1.e-20)) + MagG_B[i]
            mn = np.array([MG1**j for j in range(11)]) 
            M_B_1[i] = 10**np.sum(aML*mn)
            mn = np.array([MG2**j for j in range(11)]) 
            M_B_2[i] = 10**np.sum(aML*mn)
            M_B[i] = M_B_1[i]+M_B_2[i]

G=6.6743e-11     # Newton's constant
Msun=1.9884e30   # Msolar in kg
au=1.496e11      # AU in m
fac = 1.e-3*(1/206265)*3.086e13/3.154e7  # pc*mas/yr  to  km/s

aLM = np.loadtxt(LM,skiprows=1,unpack=True,usecols=(1),dtype=float)
def Newton_simulation(xx):
    np.random.seed(int(str(int(time.time()*1.e7))[-7:]))
    rp_sim,M_A_1_sim,M_A_2_sim,M_B_1_sim,M_B_2_sim,d_A_sim,d_A_err_sim,d_B_sim,d_B_err_sim,mux_M_sim,muy_M_sim,mux_A_err_sim,muy_A_err_sim,mux_B_err_sim,muy_B_err_sim,e_sim,e0_sim,e1_sim=xx
    ### main outer orbit: motion of two barycenters ###
    s = rp_sim  # in kau
    M_A_sim, M_B_sim = M_A_1_sim+M_A_2_sim, M_B_1_sim+M_B_2_sim
    inc = np.arccos(np.random.rand())  # random number drawn from prob(inc) = sin(inc)
    if model == 'circular':
        eps = 0
    else: # eps = eccentricity
        sige = np.random.randn()
        if sige < 0:
            eps = np.maximum(e_sim + sige*(e_sim-e0_sim),0.001)
        else:
            eps = np.minimum(e_sim + sige*(e1_sim-e_sim),0.999)
    f = lambda x: 1/(1+eps*np.cos(x))**2
    tmax, err = integrate.quad(f, 0, 2*np.pi)  # time for one complete orbit
    t = tmax*np.random.rand()
    sol=minimize(dsq,x0=[3.],args=(t,eps),bounds=[(0,2*np.pi)])  # solve the evolved azimuthal angle for t 
    phi0 = np.random.rand()*np.pi*2   # zero-point orientation of the elliptical orbit 
    phi = phi0+ sol.x[0]  # azimuthal angle
    r = 1.e3*s/np.sqrt(np.cos(phi)**2+np.cos(inc)**2*np.sin(phi)**2)  # AU
    a = r*(1+eps*np.cos(phi-phi0))/(1-eps**2)  # semi-major axis
    M = Msun*(M_A_sim+M_B_sim)
    vc = 1.e-3*np.sqrt((G*M/au)*(2/r-1/a)) # km/s
    dr_los = r*np.sin(inc)*np.sin(phi)/206265 # pc
    d_M_sim = (d_A_sim/d_A_err_sim**2+d_B_sim/d_B_err_sim**2)/(1/d_A_err_sim**2+1/d_B_err_sim**2) # initial value
    d_A_sim = d_M_sim + (M_B_sim/(M_A_sim+M_B_sim))*dr_los 
    d_B_sim = d_M_sim - (M_A_sim/(M_A_sim+M_B_sim))*dr_los 
    d_M_sim = (d_A_sim/d_A_err_sim**2+d_B_sim/d_B_err_sim**2)/(1/d_A_err_sim**2+1/d_B_err_sim**2) # simulated value
    d_M_err_sim = 1/np.sqrt(1/d_A_err_sim**2 + 1/d_B_err_sim**2)
    dv_x = -vc*np.sin(phi)
    dv_y = vc*np.cos(inc)*np.cos(phi)
    mux_A_sim = mux_M_sim + (M_B_sim/(M_A_sim+M_B_sim))*dv_x/(fac*d_A_sim) 
    mux_B_sim = mux_M_sim - (M_A_sim/(M_A_sim+M_B_sim))*dv_x/(fac*d_B_sim) 
    muy_A_sim = muy_M_sim + (M_B_sim/(M_A_sim+M_B_sim))*dv_y/(fac*d_A_sim) 
    muy_B_sim = muy_M_sim - (M_A_sim/(M_A_sim+M_B_sim))*dv_y/(fac*d_B_sim) 
    ### inner orbit(s) when close hidden component(s) is(are) present: ###
    alp = 3.5 # L ~ M^gam
    log_a_in_min = np.log10(0.01)
    log_a_in_max = np.log10(d_M_sim)
    if (M_A_2_sim != 0.) & (M_B_2_sim == 0.): # Case that A has an additional component
        M1 = M_A_1_sim
        M2 = M_A_2_sim
        Msim = M1+M2            
        #a_in = 10**((np.log10(a*1000*0.3)-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        a_in = 10**((log_a_in_max-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        gam = np.minimum(np.maximum(1 -1.26411333 + 0.85366141*np.log10(a_in),0.001),2.2)
        eps = np.random.power(gam, 1)[0]
        f = lambda x: 1/(1+eps*np.cos(x))**2
        tmax, err = integrate.quad(f, 0, 2*np.pi)
        t = tmax*np.random.rand()
        sol=minimize(dsq,x0=[3.],args=(t,eps),bounds=[(0,2*np.pi)])
        phi0 = np.random.rand()*np.pi*2
        phi = phi0+ sol.x[0]
        r_in = a_in*(1-eps**2)/(1+eps*np.cos(phi-phi0))  # in AU
        inc_CL = np.arccos(np.random.rand())
        s_in = r_in*np.sqrt(np.cos(phi)**2+np.cos(inc_CL)**2*np.sin(phi)**2)
        logM = np.log10(M2)
        mn = np.array([logM**j for j in range(11)]) 
        mag2 = np.sum(aLM*mn)+5*np.log10(d_A_sim/10)
        P_in = np.sqrt(a_in**3/(M1+M2))
        if P_in < 3:
            r_CL = 0
        else:
            r_CL = M1*M2*(M1**(alp-1)-M2**(alp-1))/((M1+M2)*(M1**alp+M2**alp)) 
        v_CL = r_CL*1.e-3*np.sqrt((G*Msun*Msim/au)*(2/r_in-1/a_in)) # km/s
        phi_CL = phi
        v_CL_x_A = -v_CL*np.sin(phi_CL)
        v_CL_y_A = v_CL*np.cos(inc_CL)*np.cos(phi_CL)
        v_CL_x_B = 0
        v_CL_y_B = 0
    elif (M_A_2_sim == 0.) & (M_B_2_sim != 0.): # Case that B has an additional component
        M1 = M_B_1_sim
        M2 = M_B_2_sim
        Msim = M1+M2            
        #a_in = 10**((np.log10(a*1000*0.3)-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        a_in = 10**((log_a_in_max-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        gam = np.minimum(np.maximum(1 -1.26411333 + 0.85366141*np.log10(a_in),0.001),2.2)
        eps = np.random.power(gam, 1)[0]
        f = lambda x: 1/(1+eps*np.cos(x))**2
        tmax, err = integrate.quad(f, 0, 2*np.pi)
        t = tmax*np.random.rand()
        sol=minimize(dsq,x0=[3.],args=(t,eps),bounds=[(0,2*np.pi)])
        phi0 = np.random.rand()*np.pi*2
        phi = phi0+ sol.x[0] 
        r_in = a_in*(1-eps**2)/(1+eps*np.cos(phi-phi0))  # in AU
        inc_CL = np.arccos(np.random.rand())
        s_in = r_in*np.sqrt(np.cos(phi)**2+np.cos(inc_CL)**2*np.sin(phi)**2)
        logM = np.log10(M2)
        mn = np.array([logM**j for j in range(11)]) 
        mag2 = np.sum(aLM*mn)+5*np.log10(d_A_sim/10)
        P_in = np.sqrt(a_in**3/(M1+M2))
        if P_in < 3:
            r_CL = 0
        else: 
            r_CL = M1*M2*(M1**(alp-1)-M2**(alp-1))/((M1+M2)*(M1**alp+M2**alp)) #0
        v_CL = r_CL*1.e-3*np.sqrt((G*Msun*Msim/au)*(2/r_in-1/a_in)) # km/s
        phi_CL = phi            
        v_CL_x_B = -v_CL*np.sin(phi_CL)
        v_CL_y_B = v_CL*np.cos(inc_CL)*np.cos(phi_CL)
        v_CL_x_A = 0
        v_CL_y_A = 0
    elif (M_A_2_sim != 0.) & (M_B_2_sim != 0.): # Case that both A and B have additional components
        M1 = M_A_1_sim
        M2 = M_A_2_sim
        Msim = M1+M2            
        #a_in = 10**((np.log10(r*1000*0.3)-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        a_in = 10**((log_a_in_max-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        gam = np.minimum(np.maximum(1 -1.26411333 + 0.85366141*np.log10(a_in),0.001),2.2)
        eps = np.random.power(gam, 1)[0]
        f = lambda x: 1/(1+eps*np.cos(x))**2
        tmax, err = integrate.quad(f, 0, 2*np.pi)
        t = tmax*np.random.rand()
        sol=minimize(dsq,x0=[3.],args=(t,eps),bounds=[(0,2*np.pi)])
        phi0 = np.random.rand()*np.pi*2
        phi = phi0+ sol.x[0] 
        r_in = a_in*(1-eps**2)/(1+eps*np.cos(phi-phi0))  # in AU
        inc_CL = np.arccos(np.random.rand())
        s_in = r_in*np.sqrt(np.cos(phi)**2+np.cos(inc_CL)**2*np.sin(phi)**2)
        logM = np.log10(M2)
        mn = np.array([logM**j for j in range(11)]) 
        mag2 = np.sum(aLM*mn)+5*np.log10(d_A_sim/10)
        P_in = np.sqrt(a_in**3/(M1+M2))
        if P_in < 3:
            r_CL = 0
        else: 
            r_CL = M1*M2*(M1**(alp-1)-M2**(alp-1))/((M1+M2)*(M1**alp+M2**alp)) #0
        v_CL = r_CL*1.e-3*np.sqrt((G*Msun*Msim/au)*(2/r_in-1/a_in)) # km/s
        phi_CL = phi
        v_CL_x_A = -v_CL*np.sin(phi_CL)
        v_CL_y_A = v_CL*np.cos(inc_CL)*np.cos(phi_CL)
        M1 = M_B_1_sim
        M2 = M_B_2_sim
        Msim = M1+M2            
        #a_in = 10**((np.log10(r*1000*0.3)-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        a_in = 10**((log_a_in_max-log_a_in_min)*np.random.rand()+log_a_in_min)  # semi-major axis in AU
        gam = np.minimum(np.maximum(1 -1.26411333 + 0.85366141*np.log10(a_in),0.001),2.2)
        eps = np.random.power(gam, 1)[0]
        f = lambda x: 1/(1+eps*np.cos(x))**2
        tmax, err = integrate.quad(f, 0, 2*np.pi)
        t = tmax*np.random.rand()
        sol=minimize(dsq,x0=[3.],args=(t,eps),bounds=[(0,2*np.pi)])
        phi0 = np.random.rand()*np.pi*2
        phi = phi0+ sol.x[0] 
        r_in = a_in*(1-eps**2)/(1+eps*np.cos(phi-phi0))  # in AU
        inc_CL = np.arccos(np.random.rand())
        s_in = r_in*np.sqrt(np.cos(phi)**2+np.cos(inc_CL)**2*np.sin(phi)**2)
        logM = np.log10(M2)
        mn = np.array([logM**j for j in range(11)]) 
        mag2 = np.sum(aLM*mn)+5*np.log10(d_A_sim/10)
        P_in = np.sqrt(a_in**3/(M1+M2))
        if P_in < 3:
            r_CL = 0
        else: 
            r_CL = M1*M2*(M1**(alp-1)-M2**(alp-1))/((M1+M2)*(M1**alp+M2**alp)) #0
        v_CL = r_CL*1.e-3*np.sqrt((G*Msun*Msim/au)*(2/r_in-1/a_in)) # km/s
        phi_CL = phi            
        v_CL_x_B = -v_CL*np.sin(phi_CL)
        v_CL_y_B = v_CL*np.cos(inc_CL)*np.cos(phi_CL)
    else:
        v_CL_x_A = 0
        v_CL_y_A = 0
        v_CL_x_B = 0
        v_CL_y_B = 0
    mux_A_sim = mux_A_sim + v_CL_x_A/(fac*d_A_sim)
    muy_A_sim = muy_A_sim + v_CL_y_A/(fac*d_A_sim)
    mux_B_sim = mux_B_sim + v_CL_x_B/(fac*d_B_sim)
    muy_B_sim = muy_B_sim + v_CL_y_B/(fac*d_B_sim)
    
    values = d_A_sim,d_B_sim,mux_A_sim,muy_A_sim,mux_B_sim,muy_B_sim
    return values
    
fw = open(outputfile,'w')  
if __name__ == "__main__":
    
    if model != 'gaia':
        xx = []
        for i in range(len(rp)):
            xx.append([
                rp[i],M_A_1[i],M_A_2[i],M_B_1[i],M_B_2[i],
                d_A[i],d_A_err[i],d_B[i],d_B_err[i],
                mux_M[i],muy_M[i],
                mux_A_err[i],muy_A_err[i],
                mux_B_err[i],muy_B_err[i],
                e[i],e0[i],e1[i]
            ])

        pool = mp.Pool(mp.cpu_count()-1)
        res = pool.map(Newton_simulation, xx)

        for i in range(len(rp)):
            d_A[i],d_B[i],mux_A[i],muy_A[i],mux_B[i],muy_B[i] = res[i]

    # ✅ FILE WRITING SHOULD ALSO BE INSIDE MAIN
    with open(outputfile,'w') as fw:
        fw.write('source_id1,source_id2,R_chance,s[kau],d1[pc],d1_err[pc],d2[pc],d2_err[pc],MagG1,MagG2,M1[Msun],M2[Msun],mu1ra[mas/yr],mu1ra_err[mas/yr],mu1dec[mas/yr],mu1dec_err[mas/yr],mu2ra[mas/yr],mu2ra_err[mas/yr],mu2dec[mas/yr],mu2dec_err[mas/yr],RV1[km/s],RV1_err[km/s],RV2[km/s],RV2_err[km/s],gal_b[deg],ruwe1,ruwe2,bp_rp1,bp_rp2,RA1[deg],DEC1[deg],RA2[deg],DEC2[deg],e,e0,e1,A_G1[mag],A_G2[mag]\n')

        for i in range(len(rp)):
            data = '%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s\n' %(
                str(source_id_A[i]),str(source_id_B[i]),str(R_chance[i]),str(rp[i]),
                str(d_A[i]),str(d_A_err[i]),str(d_B[i]),str(d_B_err[i]),
                str(MagG_A[i]),str(MagG_B[i]),str(M_A[i]),str(M_B[i]),
                str(mux_A[i]),str(mux_A_err[i]),str(muy_A[i]),str(muy_A_err[i]),
                str(mux_B[i]),str(mux_B_err[i]),str(muy_B[i]),str(muy_B_err[i]),
                str(RV_A[i]),str(RV_A_err[i]),str(RV_B[i]),str(RV_B_err[i]),
                str(gal_b[i]),str(ruwe_A[i]),str(ruwe_B[i]),
                str(bp_rp_A[i]),str(bp_rp_B[i]),
                str(ra_A[i]),str(dec_A[i]),str(ra_B[i]),str(dec_B[i]),
                str(e[i]),str(e0[i]),str(e1[i]),
                str(A_G_A[i]),str(A_G_B[i])
            )
            fw.write(data)

